#  Phân tích tỷ suất sinh lời và Quản trị rủi ro định lượng (Quantitative Analysis of Returns and Risk Management)

### Thêm đường dẫn dữ liệu

In [ ]:
# Thêm đường dẫn csv
csv_path = '/workspaces/Phan-Tich-va-Quan-Li-Dau-Tu-Nang-Cao/output/historical_stock_data_full.csv'

### Cài đặt các thư viện cần sử dụng


In [ ]:
#  Thư viện dùng để "biến hóa" các biểu đồ dữ liệu đơn điệu thành phong cách Cyberpunk (tương lai, neon, rực rỡ và hiện đại) chỉ với vài dòng code

!pip install mplcyberpunk

### Khai báo và thiết lập

In [ ]:
# Khai báo thư viện cho toàn bộ tài liệu

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import numpy as np


In [ ]:
# Import thư viện mplcyberpunk
import mplcyberpunk

In [ ]:
# Thiết lập màu nền tối và chữ neon
plt.rcParams.update({
    "axes.facecolor": "#1c1c1c",
    "figure.facecolor": "#1c1c1c",
    "axes.edgecolor": "#444444",
    "grid.color": "#444444",
    "text.color": "white",
    "axes.labelcolor": "white",
    "xtick.color": "#888888",
    "ytick.color": "#888888",
    "legend.edgecolor": "#444444",
    "legend.facecolor": "#1c1c1c",
    "legend.labelcolor": "white"
})

In [ ]:
# Xóa cache font của Matplotlib (không cần rebuild())
import matplotlib.font_manager as fm

# Cấu hình Matplotlib sử dụng font DejaVu Sans
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False # Để hiển thị dấu trừ đúng cách

### Bài làm

In [ ]:
# Kiểm tra số liệu

df_macro = pd.read_csv(csv_path)
print("5 hàng đầu tiên của DataFrame:")
display(df_macro.head())

In [ ]:
# Lọc danh mục cổ phiếu theo mã để phân tích
selected_symbols = ['FPT', 'VHC', 'HAH', 'HPG', 'HHV', 'TCB', 'MBB', 'KDH', 'KDC', 'KBC', 'POW', 'VGI', 'CMG', 'MWG', 'FRT', 'MCH']

symbol_col = next((c for c in df_macro.columns if c.lower() in ['symbol', 'ticker', 'code', 'stock']), None)
if symbol_col is None:
    raise ValueError('Không tìm thấy cột mã cổ phiếu trong df_macro. Hãy kiểm tra tên cột dữ liệu.')

df_portfolio = df_macro[df_macro[symbol_col].isin(selected_symbols)].copy()
print('Số lượng bản ghi trong danh mục:', len(df_portfolio))
print('Các mã xuất hiện trong danh mục:')
print(sorted(df_portfolio[symbol_col].unique()))

display(df_portfolio.head())

In [ ]:
# Tính chuỗi lợi nhuận theo định dạng (1+r) và quy năm cho từng cổ phiếu
df_portfolio['Date'] = pd.to_datetime(df_portfolio['Date'])
df_portfolio = df_portfolio.sort_values([symbol_col, 'Date']).reset_index(drop=True)

df_portfolio['Close_prev'] = df_portfolio.groupby(symbol_col)['Close'].shift(1)
df_portfolio['cash_return'] = (df_portfolio['Close'] - df_portfolio['Close_prev']) / df_portfolio['Close_prev']
df_portfolio['total_return'] = df_portfolio['cash_return']
df_portfolio['gross_return'] = 1 + df_portfolio['total_return']

df_portfolio['cumulative_gross'] = (
    df_portfolio.groupby(symbol_col)['gross_return']
    .cumprod()
    .where(df_portfolio['gross_return'].notna())
)

trading_days_year = 252
daily_stats = (
    df_portfolio.dropna(subset=['gross_return'])
    .groupby(symbol_col)
    .agg(
        total_period_days=('gross_return', 'count'),
        total_cumulative=('gross_return', 'prod'),
        average_daily_return=('total_return', 'mean')
    )
)
daily_stats['annualized_return'] = (
    daily_stats['total_cumulative'] ** (trading_days_year / daily_stats['total_period_days']) - 1
)
daily_stats['annualized_return_from_mean'] = (
    (1 + daily_stats['average_daily_return']) ** trading_days_year - 1
)

daily_stats = daily_stats.reset_index()
print('Lợi nhuận quy năm theo từng cổ phiếu:')
display(daily_stats)

annual_perf = (
    df_portfolio.set_index('Date')
    .groupby(symbol_col)['gross_return']
    .resample('A')
    .prod()
    .sub(1)
    .reset_index(name='annual_return')
)
annual_perf['Year'] = annual_perf['Date'].dt.year
print('Lợi nhuận thực tế theo năm cho từng cổ phiếu:')
display(annual_perf[[symbol_col, 'Year', 'annual_return']].sort_values([symbol_col, 'Year']).head(50))

In [ ]:
df_portfolio['price_change'] = df_portfolio['Close'] - df_portfolio['Close_prev']

In [ ]:
# ==============================================================================
# STEP 1: ĐỊNH NGHĨA DANH MỤC THEO NGÀNH
# ==============================================================================
sector_mapping = {
    'FPT': '1. Nhóm hưởng lợi từ hội nhập',
    'VHC': '1. Nhóm hưởng lợi từ hội nhập',
    'HAH': '1. Nhóm hưởng lợi từ hội nhập',
    'HPG': '1. Nhóm hưởng lợi từ hội nhập',
    'HHV': '1. Nhóm hưởng lợi từ hội nhập',
    
    'TCB': '2. Nhóm nhạy với chu kỳ tiền tệ',
    'MBB': '2. Nhóm nhạy với chu kỳ tiền tệ',
    'KDH': '2. Nhóm nhạy với chu kỳ tiền tệ',
    'KBC': '2. Nhóm nhạy với chu kỳ tiền tệ',
    'POW': '2. Nhóm nhạy với chu kỳ tiền tệ',
    
    'VGI': '3. Nhóm dài hạn dựa vào nội lực',
    'CMG': '3. Nhóm dài hạn dựa vào nội lực',
    'MWG': '3. Nhóm dài hạn dựa vào nội lực',
    'FRT': '3. Nhóm dài hạn dựa vào nội lực',
    'MCH': '3. Nhóm dài hạn dựa vào nội lực'
}

# Thêm cột ngành vào dataframe
df_portfolio['Sector'] = df_portfolio[symbol_col].map(sector_mapping)

# ==============================================================================
# STEP 2: CHUẨN HÓA GIÁ CLOSE VỀ GỐC 100 (BASE 100) ĐỂ DỄ SO SÁNH
# ==============================================================================
# Đảm bảo dữ liệu đã sort theo ngày để lấy giá đầu tiên chính xác
df_portfolio = df_portfolio.sort_values(['Date', symbol_col])

# Tính giá ngày đầu tiên của từng mã
first_prices = df_portfolio.groupby(symbol_col)['Close'].transform('first')

# Tạo cột giá chuẩn hóa (Giả sử ngày đầu tiên tất cả cổ phiếu đều có giá 100)
df_portfolio['Close_Normalized'] = (df_portfolio['Close'] / first_prices) * 100

In [ ]:
plt.style.use("dark_background")
# ==============================================================================
# VẼ BIỂU ĐỒ BIẾN ĐỘNG THEO TỪNG NGÀNH
# ==============================================================================

sectors = [
    '1. Nhóm hưởng lợi từ hội nhập',
    '2. Nhóm nhạy với chu kỳ tiền tệ',
    '3. Nhóm dài hạn dựa vào nội lực'
]

# Tạo bộ khung đồ thị gồm 3 hàng dọc tương ứng 3 nhóm ngành
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(15, 20), sharex=True)

# Lựa chọn palette màu sắc nét, tương phản tốt trên nền đen (không cần hiệu ứng ứng sáng)
bright_palette = sns.color_palette("bright")

for i, sector in enumerate(sectors):
    # Lọc tập dữ liệu con thuộc ngành hiện tại
    sector_data = df_portfolio[df_portfolio['Sector'] == sector]
    
    # Vẽ đường xu hướng trơn (Đã bỏ hàm make_lines_glow)
    sns.lineplot(
        data=sector_data,
        x='Date',
        y='Close_Normalized',
        hue=symbol_col,
        ax=axes[i],
        linewidth=2,
        palette="Set2", # Bảng màu tối giản, rõ nét, tách biệt màu tốt
        legend='brief'
    )
    
    # Cấu hình tiêu đề và các nhãn trục cho đồ thị con
    axes[i].set_title(f'{sector} (Mốc cơ sở ngày đầu = 100)', fontsize=14, fontweight='bold', pad=12)
    axes[i].set_ylabel('Giá chuẩn hóa (Base 100)', fontsize=11)
    axes[i].set_xlabel('Thời gian', fontsize=11)
    
    # Định dạng lưới mảnh tinh tế (màu xám nhạt mờ trên nền đen)
    axes[i].grid(True, color="#FFFEFE", linestyle='--', linewidth=0.5)

In [ ]:
# Tính toán các chỉ số cơ bản và rủi ro
advanced_stats = (
    df_portfolio.dropna(subset=['total_return'])
    .groupby(symbol_col)
    .agg(
        total_period_days=('gross_return', 'count'),
        total_cumulative=('gross_return', 'prod'),
        average_daily_return=('total_return', 'mean'),
        daily_volatility=('total_return', 'std')
    )
).reset_index()

# Quy năm chỉ số
advanced_stats['annualized_return'] = (advanced_stats['total_cumulative'] ** (252 / advanced_stats['total_period_days'])) - 1
advanced_stats['annualized_volatility'] = advanced_stats['daily_volatility'] * np.sqrt(252)
advanced_stats['sharpe_ratio'] = advanced_stats['annualized_return'] / advanced_stats['annualized_volatility']

In [ ]:
# Cấu hình theme đen
plt.style.use("dark_background")
plt.figure(figsize=(12, 6), facecolor='#000000')
plt.gca().set_facecolor('#000000')

# Sắp xếp dữ liệu theo lợi nhuận giảm dần
stats_sorted_by_return = advanced_stats.sort_values(by='annualized_return', ascending=False)

# Vẽ biểu đồ cột
sns.barplot(
    data=stats_sorted_by_return,
    x=symbol_col,
    y='annualized_return',
    palette='viridis'
)

plt.title('1. Xếp Hạng Lợi Nhuận Quy Năm (CAGR)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Mã Cổ Phiếu', fontsize=11)
plt.ylabel('Tỷ suất lợi nhuận (%)', fontsize=11)
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.grid(True, color='#262626', linestyle='--', linewidth=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Cấu hình theme đen



# Sắp xếp rủi ro từ thấp đến cao (càng thấp càng an toàn)
stats_sorted_by_vol = advanced_stats.sort_values(by='annualized_volatility', ascending=True)

# Vẽ biểu đồ cột
sns.barplot(
    data=stats_sorted_by_vol,
    x=symbol_col,
    y='annualized_volatility',
    palette='magma'
)

plt.title('2. Xếp Hạng Rủi Ro Quy Năm (Biến Động Giá - Thấp Càng Tốt)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Mã Cổ Phiếu', fontsize=11)
plt.ylabel('Mức độ biến động (%)', fontsize=11)
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.grid(True, color='#262626', linestyle='--', linewidth=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Vẽ các điểm tọa độ rủi ro - lợi nhuận
fig, ax = plt.subplots(figsize=(12, 8), facecolor='#000000')
ax.set_facecolor('#000000')

scatter = ax.scatter(
    data=advanced_stats,
    x='annualized_volatility',
    y='annualized_return',
    c='sharpe_ratio',      
    cmap='RdYlGn',         
    s=250,                 
    edgecolor='#FFFFFF',
    linewidth=1
)

# Gắn nhãn mã cổ phiếu lên từng chấm tròn
for idx, row in advanced_stats.iterrows():
    ax.text(
        x=row['annualized_volatility'] + 0.005,
        y=row['annualized_return'] + 0.005,
        s=row[symbol_col],
        fontweight='bold',
        color='#FFFFFF',
        fontsize=10
    )

# Cấu hình thanh màu Colorbar hiển thị Sharpe bên phải
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Chỉ số Sharpe (Hiệu suất trên một đơn vị rủi ro)', size=11, color='#FFFFFF')

plt.title('3. Ma Trận Tối Ưu Hóa: Lợi Nhuận vs Rủi Ro', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Rủi Ro Quy Năm (Volatility %)', fontsize=11)
plt.ylabel('Lợi Nhuận Quy Năm (Return %)', fontsize=11)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.grid(True, color='#262626', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Cấu hình theme đen
plt.style.use("dark_background")
plt.figure(figsize=(11, 8), facecolor='#000000')

# Sắp xếp các hàng cổ phiếu dựa theo thứ hạng Sharpe giảm dần
stats_sorted_by_sharpe = advanced_stats.sort_values(by='sharpe_ratio', ascending=False)
annual_pivot = annual_perf.pivot(index=symbol_col, columns='Year', values='annual_return')
annual_pivot_sorted = annual_pivot.reindex(stats_sorted_by_sharpe[symbol_col])

# Vẽ Heatmap
sns.heatmap(
    annual_pivot_sorted,
    annot=True,
    fmt=".1%",
    cmap="RdYlGn",
    center=0.0,
    linewidths=0.5,
    cbar_kws={'label': 'Lợi nhuận thực tế (%)', 'format': mtick.PercentFormatter(1.0)},
    facecolor='#000000'
)

plt.title('4. Độ Ổn Định: Hiệu Suất Sinh Lời Thực Tế Theo Năm', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Năm', fontsize=11)
plt.ylabel('Mã Cổ Phiếu (Đã xếp theo Sharpe giảm dần)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# TÍNH TOÁN MAXIMUM DRAWDOWN (MDD) CHO MỖI CỔ PHIẾU
# ==============================================================================

mdd_stats = {}

for symbol in df_portfolio[symbol_col].unique():
    symbol_data = df_portfolio[df_portfolio[symbol_col] == symbol].sort_values('Date').copy()
    
    # Chỉ sử dụng các hàng có daily return hợp lệ (không NaN)
    symbol_data = symbol_data.dropna(subset=['total_return']).copy()
    
    if len(symbol_data) == 0:
        continue
    
    # Tính cumulative growth: từ 100 đơn vị
    symbol_data['equity_curve'] = 100 * symbol_data['gross_return'].cumprod()
    
    # Tính running maximum (đỉnh cao nhất tính đến thời điểm đó)
    symbol_data['running_max'] = symbol_data['equity_curve'].expanding().max()
    
    # Tính Drawdown tại mỗi thời điểm (% so với peak)
    symbol_data['drawdown_pct'] = ((symbol_data['equity_curve'] - symbol_data['running_max']) / 
                                    symbol_data['running_max']) * 100
    
    # Lấy MDD (Drawdown tệ nhất / âm sâu nhất)
    mdd_value = symbol_data['drawdown_pct'].min()
    mdd_date = symbol_data[symbol_data['drawdown_pct'] == mdd_value]['Date'].iloc[0]
    
    # Tính thời gian phục hồi: từ khi MDD xảy ra đến khi quay lại level trước MDD
    mdd_idx = symbol_data[symbol_data['Date'] == mdd_date].index[0]
    recovery_level = symbol_data.loc[mdd_idx, 'running_max']
    
    # Tìm ngày đầu tiên sau MDD mà equity quay lại level recovery_level
    after_mdd_data = symbol_data.loc[mdd_idx+1:]
    recovery_check = after_mdd_data[after_mdd_data['equity_curve'] >= recovery_level]
    
    if len(recovery_check) > 0:
        recovery_date = recovery_check['Date'].iloc[0]
        recovery_days = int((recovery_date - mdd_date).days)
    else:
        recovery_days = None
    
    mdd_stats[symbol] = {
        'mdd_percent': mdd_value,
        'recovery_days': recovery_days
    }

# Chuyển thành DataFrame
mdd_df = pd.DataFrame(mdd_stats).T.reset_index()
mdd_df.rename(columns={'index': symbol_col}, inplace=True)

# Drop MDD columns nếu đã tồn tại trong advanced_stats (từ lần chạy trước)
cols_to_drop = [c for c in advanced_stats.columns if c in ['mdd_percent', 'recovery_days', 'recovery_gain_needed']]
if cols_to_drop:
    advanced_stats = advanced_stats.drop(columns=cols_to_drop)

# Merge với advanced_stats để có view toàn diện
advanced_stats = advanced_stats.merge(mdd_df, on=symbol_col, how='left')

print("=" * 80)
print("PHÂN TÍCH MAXIMUM DRAWDOWN (MDD) - MỨC SỤT GIẢM LỚN NHẤT")
print("=" * 80)
display(advanced_stats[[symbol_col, 'annualized_return', 'annualized_volatility', 
                        'sharpe_ratio', 'mdd_percent', 'recovery_days']].sort_values('mdd_percent'))

# Tính toán ratio hồi phục (bao nhiêu % lãi cần để hòa vốn từ MDD)
# Công thức: Nếu mất x%, cần lãi (1/(1-x%) - 1)*100 để hồi phục
advanced_stats['recovery_gain_needed'] = (1 / (1 + advanced_stats['mdd_percent']/100) - 1) * 100

print("\n" + "=" * 80)
print("LỢI NHUẬN CẦN THIẾT ĐỂ HÒA VỐN TỪ MDD")
print("=" * 80)
recovery_needed = advanced_stats[[symbol_col, 'mdd_percent', 'recovery_gain_needed']].sort_values('mdd_percent')
recovery_needed.columns = [symbol_col, 'MDD (%)', 'Lãi Cần Để Hòa Vốn (%)']
display(recovery_needed)

In [ ]:
# ==============================================================================
# VISUALIZATION: So sánh MDD vs Lợi Nhuận Quy Năm
# ==============================================================================

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(18, 7), facecolor='#000000')

# Subplot 1: Maximum Drawdown (sắp xếp từ "tốt" nhất - MDD ít nhất - đến "tệ" nhất)
stats_sorted_by_mdd = advanced_stats.sort_values('mdd_percent', ascending=False)

sns.barplot(
    data=stats_sorted_by_mdd,
    x=symbol_col,
    y='mdd_percent',
    ax=axes[0],
    palette='coolwarm'  # Đỏ (MDD sâu - tệ) -> Xanh (MDD ít - tốt)
)
axes[0].set_title('Maximum Drawdown (MDD) - Mức Sụt Giảm Tối Đa\n(Thấp Càng Tốt)', 
                   fontsize=13, fontweight='bold', pad=10, color='white')
axes[0].set_xlabel('Mã Cổ Phiếu', fontsize=11, color='white')
axes[0].set_ylabel('MDD (%)', fontsize=11, color='white')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
axes[0].grid(True, color='#262626', linestyle='--', linewidth=0.5)
axes[0].tick_params(axis='x', rotation=45, colors='white')
axes[0].tick_params(axis='y', colors='white')

# Subplot 2: Scatter - MDD vs Lợi Nhuận (Efficiency Frontier)
scatter = axes[1].scatter(
    data=advanced_stats,
    x='mdd_percent',
    y='annualized_return',
    c='sharpe_ratio',
    cmap='RdYlGn',
    s=300,
    edgecolor='#FFFFFF',
    linewidth=2,
    alpha=0.8
)

# Gắn nhãn mã cổ phiếu
for idx, row in advanced_stats.iterrows():
    axes[1].text(
        x=row['mdd_percent'] - 2,
        y=row['annualized_return'] + 0.005,
        s=row[symbol_col],
        fontweight='bold',
        color='#FFFFFF',
        fontsize=10
    )

# Thêm colorbar
cbar = fig.colorbar(scatter, ax=axes[1])
cbar.set_label('Sharpe Ratio', size=11, color='white')

axes[1].set_title('Đánh Đổi: Mức Rủi Ro (MDD) vs Lợi Nhuận\n(Góc trên-trái = Tối Ưu)', 
                  fontsize=13, fontweight='bold', pad=10, color='white')
axes[1].set_xlabel('Maximum Drawdown (%)', fontsize=11, color='white')
axes[1].set_ylabel('Lợi Nhuận Quy Năm (%)', fontsize=11, color='white')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
axes[1].grid(True, color='#262626', linestyle='--', linewidth=0.5)
axes[1].tick_params(colors='white')

plt.suptitle('5. Phân Tích Khả Năng Chịu Đựng: Maximum Drawdown (MDD)', 
             fontsize=15, fontweight='bold', color='white', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# VISUALIZATION: Drawdown Curves - Theo Dõi "Vực Thẳm" Theo Thời Gian
# ==============================================================================

# Tính toán drawdown time series cho tất cả mã
df_portfolio_sorted = df_portfolio.sort_values([symbol_col, 'Date']).copy()

# Tính equity curve từ 100 đơn vị
df_portfolio_sorted = df_portfolio_sorted.dropna(subset=['total_return'])
df_portfolio_sorted['equity_curve'] = (
    df_portfolio_sorted.groupby(symbol_col)['gross_return'].cumprod() * 100
)

df_portfolio_sorted['running_max'] = (
    df_portfolio_sorted.groupby(symbol_col)['equity_curve'].expanding().max().values
)

# Drawdown = (Current - Peak) / Peak * 100, kết quả từ 0% đến -100%
df_portfolio_sorted['drawdown'] = (
    (df_portfolio_sorted['equity_curve'] - df_portfolio_sorted['running_max']) / 
    df_portfolio_sorted['running_max']
) * 100

# Vẽ drawdown curves cho các mã được lựa chọn
fig, ax = plt.subplots(figsize=(16, 8), facecolor='#000000')
ax.set_facecolor('#000000')

# Chọn top 8 mã theo Sharpe Ratio để tránh quá tải biểu đồ
top_stocks = advanced_stats.nlargest(8, 'sharpe_ratio')[symbol_col].tolist()

colors = sns.color_palette("husl", len(top_stocks))
for stock, color in zip(top_stocks, colors):
    stock_data = df_portfolio_sorted[df_portfolio_sorted[symbol_col] == stock]
    ax.plot(stock_data['Date'], stock_data['drawdown'], label=stock, linewidth=2, color=color)

ax.axhline(y=0, color='white', linestyle='--', linewidth=1, alpha=0.5)
ax.fill_between(df_portfolio_sorted['Date'].unique(), 0, -100, alpha=0.1, color='red', label='Vùng Drawdown')
ax.set_title('6. Lịch Sử Sụt Giảm (Drawdown Curves)\nTop 8 Cổ Phiếu Theo Sharpe Ratio', 
             fontsize=14, fontweight='bold', pad=15, color='white')
ax.set_xlabel('Thời Gian', fontsize=12, color='white')
ax.set_ylabel('Drawdown (%)', fontsize=12, color='white')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.grid(True, color='#262626', linestyle='--', linewidth=0.5, alpha=0.7)
ax.legend(loc='lower left', fontsize=10, framealpha=0.9)
ax.tick_params(colors='white')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# BẢNG TỔNG HỢP COMPREHENSIVE: Xếp Hạng Bền Vững Dựa Trên MDD + Recovery Time
# ==============================================================================

# Tính điểm bền vững (Sustainability Score)
# Điểm cao = MDD ít + Recovery nhanh + Lợi nhuận cao + Sharpe cao
advanced_stats['sustainability_score'] = (
    (1 - (advanced_stats['mdd_percent'].abs() / advanced_stats['mdd_percent'].abs().max())) * 40 +  # 40 điểm: MDD
    (advanced_stats['sharpe_ratio'] / advanced_stats['sharpe_ratio'].max()) * 30 +  # 30 điểm: Sharpe
    (advanced_stats['annualized_return'] / advanced_stats['annualized_return'].max()) * 30  # 30 điểm: Return
)

# Sắp xếp theo Sustainability Score
sustainability_ranking = advanced_stats[[
    symbol_col, 'mdd_percent', 'recovery_days', 'annualized_return', 
    'annualized_volatility', 'sharpe_ratio', 'sustainability_score'
]].sort_values('sustainability_score', ascending=False).copy()

sustainability_ranking.columns = [
    'Mã CP', 'MDD (%)', 'Thời Gian Hồi Phục (ngày)', 'Lợi Nhuận (%)', 
    'Volatility (%)', 'Sharpe Ratio', 'Điểm Bền Vững'
]

print("=" * 100)
print("BẢNG XẾP HẠNG BỀN VỮNG: MDD + Thời Gian Hồi Phục + Lợi Nhuận + Sharpe")
print("=" * 100)
display(sustainability_ranking)

# Phân loại rủi ro MDD
print("\n" + "=" * 100)
print("PHÂN LOẠI RỦI RO THEO MDD")
print("=" * 100)

risk_categories = []
for _, row in advanced_stats.iterrows():
    mdd = row['mdd_percent']
    if mdd >= -15:
        category = "🟢 XANH - RỦI RO THẤP (MDD ≥ -15%)"
    elif mdd >= -25:
        category = "🟡 VÀNG - RỦI RO TRUNG BÌNH (MDD -15% đến -25%)"
    elif mdd >= -40:
        category = "🟠 CAM - RỦI RO CAO (MDD -25% đến -40%)"
    else:
        category = "🔴 ĐỎ - RỦI RO RẤT CAO (MDD < -40%)"
    
    risk_categories.append({
        'Mã CP': row[symbol_col],
        'MDD (%)': f"{row['mdd_percent']:.2f}%",
        'Phân Loại Rủi Ro': category
    })

risk_df = pd.DataFrame(risk_categories).sort_values('Mã CP')
display(risk_df)

print("\n" + "=" * 100)
print("HỎI ĐÁP VỀ MAXIMUM DRAWDOWN")
print("=" * 100)
print("""
❓ Lỗ 50% cần lãi bao nhiêu để hòa vốn?
   → Lãi 100% (số tiền vốn con lại chỉ bằng 50% số tiền ban đầu)
   
❓ Lỗ 30% cần lãi bao nhiêu?
   → Lãi ~42.9% (1 / 0.7 - 1 = 0.4286)
   
❓ Lỗ 90% cần lãi bao nhiêu?
   → Lãi 900% (1 / 0.1 - 1 = 9.0) - Hầu như bất khả thi!

🎯 KINH NGHIỆM QUẢN LÝ RỦI RO:
   1. Chọn danh mục với MDD < -30% để dễ kiểm soát tâm lý
   2. Kết hợp các mã có MDD sâu khác nhau (decorrelation)
   3. Thiết lập Stop-Loss ở mức -15% đến -20% để bảo vệ
   4. Chia dồn vốn theo từng giai đoạn để giảm rủi ro đỉnh
""")

In [ ]:
# ==============================================================================
# TÍNH TOÁN "HÌNH DÁNG" LỢI NHUẬN (SKEWNESS & KURTOSIS)
# ==============================================================================

shape_stats = (
    df_portfolio.dropna(subset=['total_return'])
    .groupby(symbol_col)['total_return']
    .agg(
        skewness='skew',       # Tính Độ lệch (Skewness)
        excess_kurtosis=lambda x: x.kurtosis() # Tính Độ nhọn dư thừa (Excess Kurtosis, chuẩn = 0)
    )
).reset_index()

# Merge vào bảng advanced_stats tổng của bạn
advanced_stats = advanced_stats.merge(shape_stats, on=symbol_col, how='left')

print("=" * 80)
print("BẢNG ĐÁNH GIÁ HÌNH DÁNG LỢI NHUẬN ĐỂ QUẢN TRỊ RỦI RO CỰC ĐOAN")
print("=" * 80)
display(advanced_stats[[symbol_col, 'annualized_return', 'mdd_percent', 'skewness', 'excess_kurtosis']].sort_values('skewness', ascending=False))

In [ ]:
plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(12, 8), facecolor='#000000')
ax.set_facecolor('#000000')

# Vẽ biểu đồ phân tán giữa Skewness và Kurtosis
scatter = ax.scatter(
    data=advanced_stats,
    x='skewness',
    y='excess_kurtosis',
    c='mdd_percent', # Tô màu chấm tròn theo Mức sụt giảm lớn nhất để thấy mối tương quan
    cmap='Reds_r',   # Màu đỏ đậm đại diện cho MDD âm rất sâu
    s=250,
    edgecolor='#FFFFFF',
    linewidth=1
)

# Gắn nhãn mã cổ phiếu
for idx, row in advanced_stats.iterrows():
    ax.text(
        x=row['skewness'] + 0.03,
        y=row['excess_kurtosis'] + 0.1,
        s=row[symbol_col],
        fontweight='bold',
        color='#FFFFFF',
        fontsize=10
    )

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Maximum Drawdown (MDD %)', size=11, color='#FFFFFF')

# Vẽ đường ranh giới lý tưởng (Giả định phân phối chuẩn có Skew = 0, Excess Kurtosis = 0)
ax.axvline(x=0, color='#555555', linestyle='--', linewidth=1)
ax.axhline(y=0, color='#555555', linestyle='--', linewidth=1)

plt.title('MA TRẬN HÌNH DÁNG LỢI NHUẬN: SKEWNESS VS KURTOSIS', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Độ lệch (Skewness) -> Càng lớn/dương càng tốt', fontsize=11)
plt.ylabel('Độ nhọn dư thừa (Excess Kurtosis) -> Càng thấp càng an toàn', fontsize=11)
plt.grid(True, color='#1a1a1a', linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.show()

#### Kết luận
##### MCH --> Hàng tiêu dùng thiết yếu --> Skewness dương, giữ tiền siêu hạng.
##### MBB --> Ngân hàng --> Sharpe tốt nhất nhóm tài chính, Drawdown nông.
##### FRT --> Bán lẻ / Dược phẩm --> CAGR 59.3%, Sharpe đỉnh bảng, phân phối lệch dương.
##### HAH --> Vận tải biển / Logistics --> CAGR Quán quân ($61\%$), chấp nhận biến động lớn để ăn sóng lớn.
##### VGI --> Viễn thông quốc tế --> Đại diện cho mảng Công nghệ/Viễn thông
##### KDH --> Bất động sản dân cư --> Skewness ổn định, né hoàn toàn bẫy KBC.
